# Prominence Oscillations — Implementation / 프로미넌스 진동 구현

**Paper**: Arregui, Oliver, & Ballester (2018), *Living Reviews in Solar Physics* 15:3
**DOI**: 10.1007/s41116-018-0012-6

This notebook reproduces central results of the review:
1. Kink mode period vs. thread length for different magnetic field strengths
2. Dispersion diagram of slow, fast, and kink modes
3. Synthetic time series of a damped resonantly-absorbed oscillation
4. Prominence seismology inversion — magnetic field from P and L
5. Large vs. small amplitude oscillations (LAO driven by flare pulse vs. SAO driven by stochastic forcing)

---

이 노트북은 리뷰의 핵심 결과를 재현한다:
1. 자기장 세기별 thread 길이에 따른 kink 모드 주기
2. slow, fast, kink 모드의 분산 다이어그램
3. 공명흡수 감쇠 진동 합성 시계열
4. 프로미넌스 지진학 역산 — 주기와 길이에서 자기장 세기 추정
5. 대진폭과 소진폭 진동 비교 (LAO는 플레어 펄스, SAO는 확률적 강제)

In [ ]:
"""Imports and physical constants used throughout the notebook."""

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import periodogram

# Physical constants (SI units)
MU_0 = 4.0 * np.pi * 1.0e-7           # Magnetic permeability of vacuum [H/m]
G_SUN = 274.0                         # Solar surface gravity [m/s^2]
K_B = 1.380649e-23                    # Boltzmann constant [J/K]
M_P = 1.67262192e-27                  # Proton mass [kg]
GAMMA = 5.0 / 3.0                     # Adiabatic index

# Typical quiescent prominence parameters (Arregui et al. 2018)
RHO_P = 5.0e-11                       # Prominence density [kg/m^3]  (~ 3e10 cm^-3)
RHO_C = 2.5e-13                       # Coronal density [kg/m^3]     (~ 1.5e8 cm^-3)
T_P = 8000.0                          # Prominence temperature [K]
T_C = 1.0e6                           # Coronal temperature [K]
ZETA = RHO_P / RHO_C                  # Density contrast

print(f'Density contrast zeta = rho_p/rho_c = {ZETA:.0f}')
print(f'Prominence sound speed = {np.sqrt(GAMMA * K_B * T_P / M_P) * 1e-3:.2f} km/s')
print(f'Coronal sound speed    = {np.sqrt(GAMMA * K_B * T_C / M_P) * 1e-3:.2f} km/s')

## 1. Kink Mode Period vs. Thread Length / 자기 플럭스 튜브 길이에 따른 Kink 주기

**English**: For a standing fundamental kink mode ($\lambda = 2L$) in an infinitely long cylindrical thread (Eq. 30–31 in the review):

$$P_k = \frac{2L}{c_k}, \qquad c_k = \sqrt{\frac{2B^2}{\mu_0(\rho_i + \rho_e)}} \approx \sqrt{2}\, v_{A,i}$$

for the high-density-contrast limit. We sweep $L$ from $10^4$ to $2 \times 10^5$ km at four magnetic-field strengths $B \in \{5, 10, 20, 30\}$ G.

**한국어**: 무한 원통 thread의 기본 standing kink 모드 ($\lambda = 2L$)에 대해, 위 식에 따라 길이 $L$을 $10^4 \sim 2\times 10^5$ km 범위에서, 네 자기장 세기 $B \in \{5, 10, 20, 30\}$ G에 대해 주기를 계산한다.

In [ ]:
def kink_period(length_m, B_tesla, rho_i=RHO_P, rho_e=RHO_C):
    """Compute standing fundamental kink period for a cylindrical thread.

    Args:
        length_m: Flux-tube length L in metres (fundamental is lambda = 2L).
        B_tesla: Magnetic-field strength in tesla.
        rho_i: Internal (prominence) density in kg/m^3.
        rho_e: External (coronal) density in kg/m^3.

    Returns:
        Period in seconds.
    """
    c_k = np.sqrt(2.0 * B_tesla**2 / (MU_0 * (rho_i + rho_e)))
    return 2.0 * length_m / c_k


length_km = np.linspace(1e4, 2e5, 300)   # thread length in km
B_values_g = [5.0, 10.0, 20.0, 30.0]     # field strengths in gauss (1 G = 1e-4 T)

fig, ax = plt.subplots(figsize=(8, 5))
for B_g in B_values_g:
    P_min = kink_period(length_km * 1e3, B_g * 1e-4) / 60.0
    ax.plot(length_km / 1e3, P_min, label=f'B = {B_g:.0f} G')

ax.set_xlabel('Flux-tube length $L$ [Mm]')
ax.set_ylabel('Fundamental kink period $P_k$ [min]')
ax.set_title('Kink period vs. thread length / Thread 길이에 따른 kink 주기')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

**Observation / 관측**: For typical $L \sim 100$ Mm and $B = 10$ G, the period is ~30 min — consistent with intermediate-period SAOs observed in quiescent prominences.

전형적 $L \sim 100$ Mm, $B = 10$ G에서 주기 ~30분으로, quiescent 프로미넌스의 중기 SAO 관측과 일치한다.

## 2. Dispersion Diagram: Slow, Fast, and Kink Modes / 분산 다이어그램

**English**: Show the three ideal-MHD mode speeds of a prominence thread as a function of magnetic field strength. At the high-density-contrast limit:
- Slow (longitudinal, field-aligned): $c_T = v_A c_s / \sqrt{v_A^2 + c_s^2}$
- Fast (transverse compressive): $c_f = \sqrt{v_A^2 + c_s^2}$
- Kink (transverse incompressible): $c_k = v_A \sqrt{2\zeta / (1 + \zeta)}$

**한국어**: 프로미넌스 thread의 이상 MHD 세 모드 속도를 자기장 세기에 대해 표시한다. 고밀도 대비 한계에서 위 공식을 사용.

In [ ]:
def alfven_speed(B_tesla, rho):
    """Return the Alfven speed v_A = B / sqrt(mu_0 rho) in m/s."""
    return B_tesla / np.sqrt(MU_0 * rho)


def sound_speed(T_kelvin):
    """Return the adiabatic sound speed c_s = sqrt(gamma k_B T / m_p) in m/s."""
    return np.sqrt(GAMMA * K_B * T_kelvin / M_P)


def fast_speed(v_A, c_s):
    """Fast magnetoacoustic speed c_f = sqrt(v_A^2 + c_s^2)."""
    return np.sqrt(v_A**2 + c_s**2)


def cusp_speed(v_A, c_s):
    """Slow / cusp (tube) speed c_T = v_A c_s / sqrt(v_A^2 + c_s^2)."""
    return v_A * c_s / np.sqrt(v_A**2 + c_s**2)


def kink_speed(v_A_i, zeta):
    """Kink speed c_k = v_A sqrt(2 zeta / (1 + zeta))."""
    return v_A_i * np.sqrt(2.0 * zeta / (1.0 + zeta))


B_g = np.linspace(1.0, 40.0, 400)
B_T = B_g * 1.0e-4
v_A = alfven_speed(B_T, RHO_P)
c_s = sound_speed(T_P)
c_f = fast_speed(v_A, c_s)
c_T = cusp_speed(v_A, c_s)
c_k = kink_speed(v_A, ZETA)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(B_g, c_k / 1e3, 'r-', label='Kink $c_k$ (transverse / 횡)', linewidth=2)
ax.plot(B_g, c_f / 1e3, 'b-', label='Fast $c_f$')
ax.plot(B_g, v_A / 1e3, 'g--', label='Alfven $v_A$')
ax.plot(B_g, c_T / 1e3, 'm-', label='Slow / cusp $c_T$ (longitudinal / 종)')
ax.axhline(c_s / 1e3, color='gray', ls=':', label=f'$c_s = {c_s/1e3:.1f}$ km/s')
ax.set_xlabel('Magnetic field strength $B$ [G]')
ax.set_ylabel('Wave speed [km/s]')
ax.set_title('MHD dispersion diagram for a prominence thread / 프로미넌스 thread MHD 분산 다이어그램')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print(f'At B = 10 G: v_A = {alfven_speed(1e-3, RHO_P)/1e3:.1f} km/s, '
      f'c_f = {fast_speed(alfven_speed(1e-3, RHO_P), c_s)/1e3:.1f} km/s, '
      f'c_k = {kink_speed(alfven_speed(1e-3, RHO_P), ZETA)/1e3:.1f} km/s, '
      f'c_T = {cusp_speed(alfven_speed(1e-3, RHO_P), c_s)/1e3:.1f} km/s')

**Key insight / 핵심 통찰**: The kink speed $c_k$ is close to $\sqrt{2}\,v_A$ in the high-density-contrast limit. The slow mode speed is nearly field-independent (set by $c_s$). These different speeds are what enable **mode discrimination** in observations: longitudinal oscillations are slow modes, transverse are kink/fast.

고밀도 대비 극한에서 $c_k \approx \sqrt{2}\,v_A$; slow 모드는 자기장에 거의 무관($c_s$에 의해 결정). 이 속도 차이가 관측에서 **모드 식별**을 가능하게 한다: 종방향 = slow, 가로 = kink/fast.

## 3. Synthetic Time Series: Damped Oscillation with Resonant Absorption / 공명흡수 감쇠 합성 시계열

**English**: Resonantly-damped kink mode produces a damped sinusoid:
$$v(t) = v_0 \sin(2\pi t / P)\exp(-t / \tau_d)$$
with damping ratio
$$\tau_d / P = (2/\pi)(R/l)(\zeta + 1)/(\zeta - 1)$$

We generate a time series for $P = 30$ min, $\tau_d/P = 3$, $v_0 = 10$ km/s, add Gaussian noise, and fit the damped-sinusoid model.

**한국어**: 공명흡수로 감쇠된 kink 모드는 감쇠 sinusoid 형태를 가진다. $P = 30$ 분, $\tau_d/P = 3$, $v_0 = 10$ km/s로 시계열을 생성하고 Gaussian 잡음 추가 후 모델을 피팅한다.

In [ ]:
from scipy.optimize import curve_fit


def damped_sinusoid(t, v0, period, tau_d, phase):
    """Damped sinusoidal oscillation model.

    Args:
        t: Time array in seconds.
        v0: Initial velocity amplitude in km/s.
        period: Oscillation period in seconds.
        tau_d: Exponential damping time in seconds.
        phase: Initial phase in radians.

    Returns:
        Velocity in km/s.
    """
    return v0 * np.sin(2.0 * np.pi * t / period + phase) * np.exp(-t / tau_d)


def damping_ratio_resonant(R_over_l, zeta):
    """Thin-tube-thin-boundary damping ratio (Eq. 37 in Arregui+18).

    Args:
        R_over_l: Ratio of thread radius to transverse inhomogeneity scale.
        zeta: Density contrast rho_p / rho_c.

    Returns:
        tau_d / P (dimensionless).
    """
    return (2.0 / np.pi) * R_over_l * (zeta + 1.0) / (zeta - 1.0)


# Physical parameters
P_true = 30.0 * 60.0                  # 30 min in seconds
tau_over_P = damping_ratio_resonant(R_over_l=5.0, zeta=ZETA)  # l/R = 0.2
tau_true = tau_over_P * P_true
v0_true = 10.0                        # km/s
noise_std = 0.8                       # km/s

print(f'Resonant damping ratio tau_d/P = {tau_over_P:.2f}')
print(f'Damping time tau_d = {tau_true/60:.1f} min')

# Generate synthetic observations
rng = np.random.default_rng(seed=42)
t = np.arange(0, 4.0 * P_true, 30.0)     # 4 periods, 30-s cadence
v_clean = damped_sinusoid(t, v0_true, P_true, tau_true, phase=0.3)
v_obs = v_clean + rng.normal(0, noise_std, size=len(t))

# Fit to observations
p0 = (8.0, 25.0 * 60.0, 2.0 * P_true, 0.0)
popt, _ = curve_fit(damped_sinusoid, t, v_obs, p0=p0)
v0_fit, P_fit, tau_fit, phi_fit = popt

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t / 60.0, v_obs, 'k.', markersize=3, label='Synthetic observation / 합성 관측', alpha=0.6)
ax.plot(t / 60.0, v_clean, 'r--', label=f'True: P={P_true/60:.1f} min, tau_d/P={tau_over_P:.2f}')
ax.plot(t / 60.0, damped_sinusoid(t, *popt), 'b-',
        label=f'Fit: P={P_fit/60:.2f} min, tau_d/P={tau_fit/P_fit:.2f}')
ax.set_xlabel('Time [min]')
ax.set_ylabel('Velocity amplitude [km/s]')
ax.set_title('Resonantly-damped kink oscillation / 공명흡수 감쇠 kink 진동')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print('\nRecovered parameters / 회수 파라미터:')
print(f'  v0 = {v0_fit:.2f} km/s (true {v0_true})')
print(f'  P  = {P_fit/60:.2f} min (true {P_true/60:.1f})')
print(f'  tau_d = {tau_fit/60:.2f} min (true {tau_true/60:.1f})')
print(f'  tau_d/P = {tau_fit/P_fit:.2f} (true {tau_over_P:.2f})')

**English**: The fit recovers period, damping time, and ratio to within a few percent — demonstrating that typical observational cadence and noise permit reliable determination of seismic parameters.

**한국어**: 피팅이 주기·감쇠시간·감쇠비를 수 퍼센트 오차 내로 회수한다. 전형적 관측 간격과 잡음에서 지진학 파라미터의 신뢰성 있는 결정이 가능함을 보여준다.

## 4. Prominence Seismology: Magnetic Field from P and L / 지진학 자기장 역산

**English**: Given the observed standing kink period $P$ and flux-tube length $L$ (from e.g. H-alpha imaging), invert the kink-mode dispersion to obtain $B$:

$$B = \frac{L}{P}\sqrt{2\mu_0 \rho_0 \left(1 + \frac{\rho_e}{\rho_0}\right)} \quad \text{(Eq. 44)}$$

We apply this to several LAO observations cited in the review:
- **Xue et al. (2014)**: P = 1140 s, L ~ 50 Mm, $\rho_0 = 5\times 10^{-11}$ kg/m³ → B ~ 17.6 G
- **Luna et al. (2014)**: P = 50 min, R_dip = 50 Mm → B_min = 14 G (pendulum)
- **Liu et al. (2012)**: P = 28 min, L ~ 100 Mm → global kink

**한국어**: 관측된 standing kink 주기 $P$와 플럭스 튜브 길이 $L$에서 위 공식으로 $B$를 역산한다. 위 LAO 관측들에 적용한다.

In [ ]:
def seismic_B(P_sec, L_m, rho_i=RHO_P, rho_e=RHO_C):
    """Infer magnetic field strength from standing kink-mode period (Eq. 44).

    Args:
        P_sec: Oscillation period in seconds.
        L_m: Flux-tube length in metres.
        rho_i: Prominence (internal) density in kg/m^3.
        rho_e: Coronal (external) density in kg/m^3.

    Returns:
        Magnetic-field strength in tesla.
    """
    return (L_m / P_sec) * np.sqrt(2.0 * MU_0 * rho_i * (1.0 + rho_e / rho_i))


def pendulum_B_min(P_sec, n_cm3=1.0e11):
    """Minimum magnetic field from longitudinal pendulum model (Eq. 5).

    Args:
        P_sec: Longitudinal oscillation period in seconds.
        n_cm3: Prominence number density in cm^-3.

    Returns:
        Minimum magnetic-field strength in tesla.
    """
    n_m3 = n_cm3 * 1.0e6
    m = M_P
    return np.sqrt(G_SUN**2 * m * n_m3 / (4.0 * np.pi**2)) * P_sec / np.sqrt(MU_0)


# Observations from the review
observations = [
    ('Xue et al. (2014)', 1140.0, 50.0e6, 5.0e-11, 2.5e-13),
    ('Liu et al. (2012)', 28 * 60.0, 100.0e6, 5.0e-11, 2.5e-13),
    ('Gosain & Foullon (2012)', 28 * 60.0, 80.0e6, 5.0e-11, 2.5e-13),
    ('Hershaw et al. (2011)', 100 * 60.0, 200.0e6, 5.0e-11, 2.5e-13),
]

print(f'{"Event":<32} {"P (s)":>8} {"L (Mm)":>8} {"B (G)":>8}')
print('-' * 60)
for name, P, L, rho_i, rho_e in observations:
    B = seismic_B(P, L, rho_i, rho_e) * 1e4
    print(f'{name:<32} {P:>8.0f} {L/1e6:>8.0f} {B:>8.1f}')

# Pendulum: longitudinal LAO seismology
P_long = 57 * 60.0                    # Luna et al. (2014)
B_pend = pendulum_B_min(P_long) * 1e4
print(f'\nLuna et al. (2014) longitudinal LAO:')
print(f'  P = {P_long/60:.0f} min -> B_min (pendulum) = {B_pend:.1f} G')
print(f'  Radius of curvature R = {G_SUN * (P_long/(2*np.pi))**2 / 1e6:.1f} Mm')

In [ ]:
# Parameter-space diagnostic: how inferred B depends on density assumption
P_obs = 1800.0                        # 30 min
rho_i_range = np.logspace(-12, -10, 200)  # 10^-12 to 10^-10 kg/m^3

fig, ax = plt.subplots(figsize=(8, 5))
for L_Mm in [50.0, 100.0, 150.0, 200.0]:
    B_G = seismic_B(P_obs, L_Mm * 1e6, rho_i_range, RHO_C) * 1e4
    ax.loglog(rho_i_range, B_G, label=f'L = {L_Mm:.0f} Mm')

ax.set_xlabel('Assumed prominence density $\\rho_p$ [kg m$^{-3}$]')
ax.set_ylabel('Inferred magnetic field $B$ [G]')
ax.set_title(f'Seismology inversion for P = {P_obs/60:.0f} min / P = {P_obs/60:.0f}분 지진학 역산')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print('Note: The inferred B scales as sqrt(rho_p), so a factor-10 density uncertainty')
print('translates into a factor-~3 uncertainty in B. This is why Bayesian methods')
print('that propagate density uncertainty are valuable.')
print()
print('참고: 추정 B는 sqrt(rho_p)에 비례하므로 밀도 10배 불확실성은 B의 ~3배 불확실성.')
print('이 때문에 밀도 불확실성을 전파하는 Bayesian 기법이 가치 있다.')

## 5. Large vs. Small Amplitude Oscillations / 대진폭 대 소진폭 진동

**English**: LAOs are driven by impulsive energetic events (flares, EUV waves) that deposit energy $E \sim 10^{19} - 10^{20}$ J into the prominence, producing velocities > 20 km/s that shake the whole structure. SAOs are driven by continuous stochastic forcing (photospheric/chromospheric motions) at $\sim$ mHz frequencies, producing v < 3 km/s local oscillations.

We simulate two time series:
- **LAO**: impulse $\delta(t)$ → single damped kink oscillation with large amplitude
- **SAO**: Gaussian white-noise forcing → continuous low-amplitude response at the natural frequency

**한국어**: LAO는 플레어·EUV파 등 충격적 에너지 사건($E \sim 10^{19} - 10^{20}$ J)이 구동, v > 20 km/s로 전체 구조를 흔듦. SAO는 광구/채층 운동의 확률적 강제가 mHz 근처로 구동, v < 3 km/s 국소 진동. 두 시계열을 시뮬레이션한다.

In [ ]:
def driven_harmonic_oscillator(t, force, omega0, gamma):
    """Solve a damped harmonic oscillator driven by an arbitrary forcing.

    Equation: d^2x/dt^2 + 2 gamma dx/dt + omega0^2 x = force(t)

    Uses simple first-order Euler integration of the two-state ODE.

    Args:
        t: Time grid in seconds (uniform spacing).
        force: Driving force array at each time sample (km/s^2).
        omega0: Natural angular frequency [rad/s].
        gamma: Damping rate [rad/s]; equivalent tau_d = 1/gamma.

    Returns:
        Position (velocity amplitude) array in km/s.
    """
    dt = t[1] - t[0]
    x = np.zeros_like(t)
    v = np.zeros_like(t)
    for i in range(1, len(t)):
        a = force[i - 1] - 2.0 * gamma * v[i - 1] - omega0**2 * x[i - 1]
        v[i] = v[i - 1] + a * dt
        x[i] = x[i - 1] + v[i] * dt
    return x


# Common parameters
P0 = 30.0 * 60.0                      # 30-min natural period
omega0 = 2.0 * np.pi / P0
gamma = 1.0 / (3.0 * P0)              # tau_d = 3P

t = np.arange(0, 6.0 * P0, 10.0)      # 6 periods at 10-s cadence

# Large amplitude: flare pulse at t = 500 s
force_LAO = np.zeros_like(t)
pulse_idx = np.argmin(np.abs(t - 500.0))
force_LAO[pulse_idx] = 1.5            # strong impulse
x_LAO = driven_harmonic_oscillator(t, force_LAO, omega0, gamma)
# Scale to LAO amplitude
x_LAO *= 30.0 / np.max(np.abs(x_LAO))

# Small amplitude: stochastic broadband forcing
rng = np.random.default_rng(seed=7)
force_SAO = rng.normal(0.0, 2e-4, size=len(t))   # weak white-noise forcing
x_SAO = driven_harmonic_oscillator(t, force_SAO, omega0, gamma)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(t / 60.0, x_LAO, 'r-')
ax1.axvline(500.0 / 60.0, color='gray', ls=':', label='Flare pulse / 플레어 펄스')
ax1.set_ylabel('LAO velocity [km/s]')
ax1.set_title('Large Amplitude Oscillation: flare-driven damped kink / 대진폭: 플레어 구동 감쇠')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(t / 60.0, x_SAO, 'b-')
ax2.set_xlabel('Time [min]')
ax2.set_ylabel('SAO velocity [km/s]')
ax2.set_title('Small Amplitude Oscillation: stochastically forced / 소진폭: 확률적 구동')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'LAO peak velocity: {np.max(np.abs(x_LAO)):.1f} km/s')
print(f'SAO typical velocity: {np.std(x_SAO):.3f} km/s')
print(f'Amplitude ratio LAO/SAO: {np.max(np.abs(x_LAO)) / max(np.std(x_SAO), 1e-9):.0f}')

In [ ]:
# Power spectrum of the two time series: both concentrate at the natural frequency
f_LAO, Pxx_LAO = periodogram(x_LAO, fs=1.0 / (t[1] - t[0]))
f_SAO, Pxx_SAO = periodogram(x_SAO, fs=1.0 / (t[1] - t[0]))

fig, ax = plt.subplots(figsize=(8, 5))
# Convert to period (min) on the x-axis (skip zero frequency)
period_min_LAO = 1.0 / f_LAO[1:] / 60.0
period_min_SAO = 1.0 / f_SAO[1:] / 60.0
ax.semilogx(period_min_LAO, Pxx_LAO[1:] / np.max(Pxx_LAO[1:]), 'r-', label='LAO (normalised)')
ax.semilogx(period_min_SAO, Pxx_SAO[1:] / np.max(Pxx_SAO[1:]), 'b-', label='SAO (normalised)', alpha=0.7)
ax.axvline(P0 / 60.0, color='gray', ls='--', label=f'Natural period {P0/60:.0f} min')
ax.set_xlabel('Period [min]')
ax.set_ylabel('Normalised power')
ax.set_title('Power spectra: both LAO and SAO peak at the natural period / 둘 다 고유주기에서 peak')
ax.set_xlim(5, 200)
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

**English**: Both LAO and SAO pick out the same natural period ~30 min because they are both **normal-mode responses of the same structure**; the difference is only in driver amplitude and coherence. This is the conceptual basis of seismology: the observed period encodes the intrinsic magnetic/plasma structure, independent of the driver. Amplitudes and transients distinguish LAO vs. SAO.

**한국어**: LAO와 SAO 모두 같은 ~30분 고유주기를 포착한다. 이들은 동일 구조의 **정상 모드 응답**이기 때문이며, 차이는 구동 진폭·coherence일 뿐이다. 이것이 지진학의 개념적 기반이다: 관측 주기가 구동자와 무관하게 자기장·플라즈마 구조를 인코딩한다. 진폭·과도현상만이 LAO와 SAO를 구분한다.

## 6. Summary / 요약

**English**: We have implemented key building blocks of Arregui et al. (2018):

1. **Kink period formula** — $P_k$ vs. $L$ for different $B$, matches observational periods (5 min to 1 h)
2. **MHD dispersion diagram** — slow, Alfven, fast, and kink speeds map directly to observed polarisation classes
3. **Damped synthetic time series** — recovery of $P$ and $\tau_d/P$ from noisy data validates observational pipelines
4. **Seismology inversion** — Eq. (44) applied to four LAO observations gives $B \sim 10$–30 G; pendulum model gives $B_\mathrm{min} \sim 15$ G
5. **LAO vs. SAO simulation** — same natural period, different drivers demonstrate that seismology is driver-independent

These tools are prerequisites for the Bayesian inversion framework (Sect. 8.7 of the review), which extends these point estimates with rigorous uncertainty quantification and model comparison.

**한국어**: Arregui et al. (2018)의 핵심 구성요소를 구현했다:

1. **Kink 주기 공식** — $L$과 $B$에 따른 $P_k$, 관측 주기(5분–1시간)와 일치
2. **MHD 분산 다이어그램** — slow, Alfven, fast, kink 속도가 관측된 편광 범주에 직접 대응
3. **감쇠 합성 시계열** — 잡음 데이터에서 $P, \tau_d/P$ 회수가 관측 파이프라인 검증
4. **지진학 역산** — 식 (44)를 네 LAO 관측에 적용하여 $B \sim 10$–30 G; 진자 모델로 $B_\mathrm{min} \sim 15$ G
5. **LAO 대 SAO 시뮬레이션** — 같은 고유주기, 다른 구동; 지진학의 구동 독립성 증명

이 도구들은 리뷰 Sect. 8.7의 Bayesian 역산 프레임워크의 전제이며, 그 프레임워크는 이 점 추정을 엄밀한 불확실성 정량화·모델 비교로 확장한다.